# Phase 3 · Submission S6 — Knowledge Distillation: S7 (12-layer) teacher → 6-layer student

**Purpose:** the knowledge-distillation (KD) experiment required in every phase. A large, strong *teacher* (S7, best Phase 3 model: 12-layer Architecture A + LoRA + BiLSTM-CRF, Codabench 0.5019) teaches a smaller, faster *student*.

**Student.** Architecture A cut to **6 of its 12 decoder layers**, keeping every other layer (0, 2, 4, 6, 8, 10) of the Phase-1 model — the layer-selection initialisation used by DistilBERT (Sanh et al., 2019). Same LoRA (attention + FFN) and the same BiLSTM-CRF head as S7. About two-thirds of the teacher's size and roughly twice as fast at encoding sentences.

**Distillation (Hinton et al., 2015).** The teacher is run once over every training document and its per-sentence label scores are turned into *soft labels* with temperature T = 2 (softmax(scores / T)). The student is trained on

`loss = CRF NLL + class-weighted cross-entropy (true labels) + KD_WEIGHT · T² · KL(teacher soft labels ‖ student soft labels)`

so it learns both the gold labels and the teacher's "dark knowledge" (which wrong roles the teacher considers plausible).

**Two stages, matching the teacher.** Stage 1: teacher = S7's Stage-1 model (trained on train only), student trained on train, epoch and decoding (Viterbi vs argmax) chosen by dev Macro-F1 — honest dev score, and a fair teacher-vs-student comparison on dev. Stage 2: teacher = S7's Stage-2 model (train + dev), student retrained on train + dev for that many epochs → `submission.zip`; Stage 1's predictions → `submission_stage1.zip`.

**Reported for the analysis:** teacher vs student dev Macro-F1 (how much is kept), parameter counts, and the time each needs to label the dev set.

**Before running (Kaggle)**
- Accelerator: **GPU T4** (one GPU is used).
- Add-ons → Secrets: **`HF`** = your Hugging Face token.
- Add Input → datasets: **`rohitbatra2004/phase111`** (Architecture A), **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer) and **the dataset holding S7's `checkpoints/stage1/best.pt` and `checkpoints/stage2/best.pt`** (teacher).
- Expected time ≈ 1.5 hours. Checkpoints are saved every epoch for both stages; to resume after an interruption, add the previous version's output of this notebook as an input and run again.

Outputs are written to `/kaggle/working/P3_S6_KD_S7Teacher_6LayerStudent/`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P3_S6_KD_S7Teacher_6LayerStudent"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR / "stage1", CKPT_DIR / "stage2", OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Sentence encoding: 128 tokens cover ~98% of sentences (see data analysis).
MAX_SENT_TOKENS = 128
# Training windows of consecutive sentences, and sentences per forward pass at inference.
WINDOW_SENTS = 128
WINDOW_STRIDE = 64
ENCODE_CHUNK = 256

# LoRA settings — identical to the S7 teacher (needed to rebuild it, and reused for the student).
LORA_RANK = 8
LORA_ALPHA = 16.0
LORA_DROPOUT = 0.05
LORA_ATTN_TARGETS = ("q_proj", "k_proj", "v_proj", "out_proj")
LORA_FFN_TARGETS = ("fc1", "fc2")

# Student: keep every other decoder layer of the Phase-1 model (DistilBERT-style initialisation).
STUDENT_LAYERS = (0, 2, 4, 6, 8, 10)

# Head + CRF (same as S7).
SENT_DIM = 512
DOC_LSTM_HIDDEN = 256
DROPOUT = 0.2
CE_WEIGHT = 1.0

# Knowledge distillation.
KD_TEMPERATURE = 2.0
KD_WEIGHT = 1.0

# Optimisation (same as S7).
EPOCHS = 20
LR_ADAPTER = 3e-4
LR_HEAD = 1e-3
LR_CRF = 1e-2
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
PATIENCE = 5
CLASS_WEIGHT_POWER = 0.5
CLASS_WEIGHT_MAX = 10.0
USE_AMP = DEVICE.type == "cuda"

print("Run:", RUN_NAME)
print("Outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Download the Phase 3 data from Hugging Face using the Kaggle secret "HF".
# Only the Phase_3_RR folder is fetched; files already present are not downloaded again.
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=HF_TOKEN,
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_3_RR/**"])

P3_DIR = DATA_DIR / "Phase_3_RR"
REQUIRED_FILES = ["Train/CL_train.json", "Train/CL_dev.json", "Train/IT_train.json",
                  "Train/IT_dev.json", "Train/label_vocab.json",
                  "Test/CL_test.json", "Test/IT_test.json"]
missing = [f for f in REQUIRED_FILES if not (P3_DIR / f).is_file()]
assert not missing, f"Missing Phase 3 files: {missing}"
print("Phase 3 data ready in", P3_DIR)

In [ ]:
# Cell 4 — Load train/dev/test documents and the official label vocabulary.
# Each document becomes {id, domain, sents, labels}; test documents have labels = None.
LABEL2ID = json.load(open(P3_DIR / "Train/label_vocab.json"))
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
DOMAINS = {"CL": 0, "IT": 1}

def load_labelled(path, domain):
    docs = []
    for d in json.load(open(path)):
        assert len(d["text"]) == len(d["labels"]), f"Length mismatch in {d['id']}"
        docs.append({"id": d["id"], "domain": domain, "sents": d["text"],
                     "labels": [LABEL2ID[label] for label in d["labels"]]})
    return docs

def load_test(path, domain):
    return [{"id": doc_id, "domain": domain, "sents": sents, "labels": None}
            for doc_id, sents in json.load(open(path)).items()]

train_docs = load_labelled(P3_DIR / "Train/CL_train.json", "CL") + load_labelled(P3_DIR / "Train/IT_train.json", "IT")
dev_docs = load_labelled(P3_DIR / "Train/CL_dev.json", "CL") + load_labelled(P3_DIR / "Train/IT_dev.json", "IT")
test_docs = load_test(P3_DIR / "Test/CL_test.json", "CL") + load_test(P3_DIR / "Test/IT_test.json", "IT")

for name, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    print(f"{name:5s}: {len(docs):3d} documents, {sum(len(d['sents']) for d in docs):6d} sentences")

In [ ]:
# Cell 5 — Load the shared Phase-1 BPE tokenizer and turn every sentence into token ids.
# Each sentence becomes [BOS] + at most 126 BPE tokens + [EOS]; [PAD] (id 0) is used for padding.
tokenizer_paths = sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))
assert tokenizer_paths, "Add the dataset rohitbatra2004/cs781-phase1-tokenized-corpus as an input."
tokenizer = Tokenizer.from_file(tokenizer_paths[0])

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")
assert (PAD_ID, BOS_ID, EOS_ID) == (0, 2, 3), "Unexpected special-token ids"
assert tokenizer.get_vocab_size() == 50000, "Unexpected vocabulary size"

def encode_sentences(sents):
    encodings = tokenizer.encode_batch(sents)
    return [[BOS_ID] + e.ids[:MAX_SENT_TOKENS - 2] + [EOS_ID] for e in encodings]

for d in train_docs + dev_docs + test_docs:
    d["ids"] = encode_sentences(d["sents"])

lengths = [len(x) for d in train_docs for x in d["ids"]]
print("Tokenizer:", tokenizer_paths[0])
print(f"Train sentence length: mean {np.mean(lengths):.1f} tokens; "
      f"{100 * np.mean([n == MAX_SENT_TOKENS for n in lengths]):.1f}% truncated at {MAX_SENT_TOKENS}")

In [ ]:
# Cell 6 — Phase-1 Architecture A classes and the pretrained weights (kept in memory for both stages).
# The model is built from these weights twice: once for Stage 1 and once, fresh, for Stage 2.
a_paths = sorted(glob.glob("/kaggle/input/**/architecture_A/model_A_state_dict.pt", recursive=True))
assert a_paths, "Add the dataset rohitbatra2004/phase111 as an input."
A_ROOT = Path(a_paths[0]).parent
A_METADATA = json.load(open(A_ROOT / "model_A_metadata.json"))
A_CONFIG = A_METADATA["model_config"]
assert A_CONFIG["pre_norm"] and A_CONFIG["tie_embeddings"], "Phase-1 A is a pre-LN decoder with tied embeddings"

class CausalSelfAttention(nn.Module):
    """Same parameters as Phase 1; uses PyTorch's fused causal attention for speed and memory."""
    def __init__(self, cfg):
        super().__init__()
        d = cfg["d_model"]
        self.n_heads, self.dropout = cfg["n_heads"], cfg["dropout"]
        self.q_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.k_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.v_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.out_proj = nn.Linear(d, d, bias=cfg["use_bias"])
        self.resid_dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        B, T, C = x.shape
        split = lambda t: t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2)
        out = F.scaled_dot_product_attention(split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x)),
                                             dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.resid_dropout(self.out_proj(out.transpose(1, 2).reshape(B, T, C)))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["use_bias"])
        self.fc2 = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["use_bias"])
        self.dropout = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["d_model"])
        self.attention = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["d_model"])
        self.feed_forward = FeedForward(cfg)

    def forward(self, x):
        x = x + self.attention(self.ln_1(x))
        return x + self.feed_forward(self.ln_2(x))

class ArchitectureABackbone(nn.Module):
    """Phase-1 decoder-only LM without the vocabulary projection.
    Returns the final-normalised state of every token: [batch, tokens, d_model]."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = nn.LayerNorm(cfg["d_model"])
        # Tied output weight, kept so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_weight = self.token_embedding.weight

    def forward(self, input_ids):
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))
        for block in self.blocks:
            x = block(x)
        return self.final_norm(x)

saved = torch.load(a_paths[0], map_location="cpu", weights_only=True)
# The export is either the raw state dict or a dict that wraps it under "state_dict".
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved

check = ArchitectureABackbone(A_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(A_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(A_CONFIG, indent=2))
print(f"Phase-1 Architecture A weights verified from {A_ROOT} — {n_backbone:,} parameters")

In [ ]:
# Cell 7 — LoRA, and a function that returns a freshly adapted Phase-1 backbone with all or selected layers.
# make_lora_backbone() = 12-layer teacher backbone; make_lora_backbone(STUDENT_LAYERS) = 6-layer student backbone.
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        self.base = base
        self.lora_A = nn.Linear(base.in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, base.out_features, bias=False)
        self.lora_dropout = nn.Dropout(dropout)
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_B(self.lora_A(self.lora_dropout(x)))

def make_lora_backbone(layer_ids=None):
    bb = ArchitectureABackbone(A_CONFIG)
    bb.load_state_dict(PHASE1_STATE, strict=True)
    if layer_ids is not None:
        bb.blocks = nn.ModuleList([bb.blocks[i] for i in layer_ids])   # keep only the selected Phase-1 layers
    for p in bb.parameters():
        p.requires_grad_(False)
    for block in bb.blocks:
        for name in LORA_ATTN_TARGETS:
            setattr(block.attention, name, LoRALinear(getattr(block.attention, name), LORA_RANK, LORA_ALPHA, LORA_DROPOUT))
        for name in LORA_FFN_TARGETS:
            setattr(block.feed_forward, name, LoRALinear(getattr(block.feed_forward, name), LORA_RANK, LORA_ALPHA, LORA_DROPOUT))
    return bb

# Check that the student's kept layers are exact copies of the chosen Phase-1 layers.
_student_bb = make_lora_backbone(STUDENT_LAYERS)
for new_index, old_index in enumerate(STUDENT_LAYERS):
    assert torch.equal(_student_bb.blocks[new_index].ln_1.weight, PHASE1_STATE[f"blocks.{old_index}.ln_1.weight"])
    assert torch.equal(_student_bb.blocks[new_index].attention.q_proj.base.weight,
                       PHASE1_STATE[f"blocks.{old_index}.attention.q_proj.weight"])
n_student_backbone = sum(p.numel() for p in _student_bb.parameters() if not p.requires_grad)
print(f"Teacher backbone: {A_CONFIG['n_layers']} layers, {n_backbone:,} Phase-1 parameters")
print(f"Student backbone: {len(STUDENT_LAYERS)} layers {STUDENT_LAYERS}, {n_student_backbone:,} Phase-1 parameters "
      f"({100 * n_student_backbone / n_backbone:.1f}% of the teacher)")
del _student_bb

In [ ]:
# Cell 8 — CRF layer and the full model (S7 head + CRF); build_teacher() = S7 architecture, build_model() = student.
# The CRF scores a label sequence as sum(emission scores) + sum(transition scores) and decodes with Viterbi.
class LinearChainCRF(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(num_labels, num_labels))   # [previous label, next label]
        self.start = nn.Parameter(torch.zeros(num_labels))
        self.end = nn.Parameter(torch.zeros(num_labels))

    def neg_log_likelihood(self, emissions, labels):
        """emissions [S, K], labels [S] for one sequence -> -log p(labels | sentences)."""
        gold = self.start[labels[0]] + emissions[0, labels[0]]
        gold = gold + (self.transitions[labels[:-1], labels[1:]] + emissions[torch.arange(1, len(labels), device=emissions.device), labels[1:]]).sum()
        gold = gold + self.end[labels[-1]]
        alpha = self.start + emissions[0]                       # forward algorithm in log space
        for t in range(1, emissions.size(0)):
            alpha = torch.logsumexp(alpha.unsqueeze(1) + self.transitions, dim=0) + emissions[t]
        log_partition = torch.logsumexp(alpha + self.end, dim=0)
        return log_partition - gold

    @torch.no_grad()
    def viterbi(self, emissions):
        """emissions [S, K] -> best label sequence (list of ints)."""
        score = self.start + emissions[0]
        backpointers = []
        for t in range(1, emissions.size(0)):
            total = score.unsqueeze(1) + self.transitions          # [previous, next]
            score, best_prev = total.max(dim=0)
            score = score + emissions[t]
            backpointers.append(best_prev)
        best = [int((score + self.end).argmax())]
        for best_prev in reversed(backpointers):
            best.append(int(best_prev[best[-1]]))
        return best[::-1]

class RhetoricalRoleCRFModel(nn.Module):
    def __init__(self, backbone, hidden_size):
        super().__init__()
        self.backbone = backbone
        self.sent_proj = nn.Sequential(nn.Linear(2 * hidden_size, SENT_DIM), nn.GELU(), nn.Dropout(DROPOUT))
        self.domain_emb = nn.Embedding(len(DOMAINS), SENT_DIM)
        self.doc_lstm = nn.LSTM(SENT_DIM + 1, DOC_LSTM_HIDDEN, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(2 * DOC_LSTM_HIDDEN, NUM_LABELS)
        self.crf = LinearChainCRF(NUM_LABELS)

    def encode_sentences(self, ids, mask):
        """ids, mask: [S, T] -> sentence vectors [S, SENT_DIM] (mean state ; [EOS] state)."""
        states = self.backbone(ids)
        m = mask.unsqueeze(-1).to(states.dtype)
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1.0)
        last = states[torch.arange(states.size(0), device=states.device), mask.sum(1) - 1]
        return self.sent_proj(torch.cat([mean, last], dim=-1))

    def emissions(self, sent_vecs, positions, domain_id):
        """sent_vecs: [S, SENT_DIM] of one document (or window) -> per-label scores [S, NUM_LABELS]."""
        x = sent_vecs + self.domain_emb.weight[domain_id]
        x = torch.cat([x, positions.unsqueeze(-1).to(x.dtype)], dim=-1).unsqueeze(0)
        context, _ = self.doc_lstm(x)
        return self.classifier(self.dropout(context.squeeze(0)))

def build_teacher():
    """The S7 architecture (12 layers); its trained LoRA/head/CRF weights are loaded in Cell 11."""
    torch.manual_seed(SEED)
    return RhetoricalRoleCRFModel(make_lora_backbone(), A_CONFIG["d_model"]).to(DEVICE)

def build_model():
    """A fresh student: 6 Phase-1 layers + new LoRA + new head + new CRF, on the device."""
    torch.manual_seed(SEED)
    return RhetoricalRoleCRFModel(make_lora_backbone(STUDENT_LAYERS), A_CONFIG["d_model"]).to(DEVICE)

# A quick CRF self-check on random scores: the NLL must be positive and Viterbi must return one label per sentence.
_crf = LinearChainCRF(NUM_LABELS)
_em, _lab = torch.randn(7, NUM_LABELS), torch.randint(0, NUM_LABELS, (7,))
assert _crf.neg_log_likelihood(_em, _lab) > 0 and len(_crf.viterbi(_em)) == 7
del _crf, _em, _lab
print("CRF and model definitions ready.")

In [ ]:
# Cell 9 — Helpers: padding, relative positions, training windows and class weights.
# Windows of 128 consecutive sentences (stride 64); class weights come from the training labels.
def pad_batch(id_lists):
    width = max(len(x) for x in id_lists)
    ids = torch.full((len(id_lists), width), PAD_ID, dtype=torch.long)
    for i, x in enumerate(id_lists):
        ids[i, :len(x)] = torch.tensor(x)
    return ids, ids.ne(PAD_ID)

def rel_positions(n):
    return torch.arange(n, dtype=torch.float) / max(n - 1, 1)

def make_windows(docs):
    windows = []
    for doc_index, d in enumerate(docs):
        n = len(d["ids"])
        starts = list(range(0, max(n - WINDOW_SENTS, 0) + 1, WINDOW_STRIDE))
        if starts[-1] + WINDOW_SENTS < n:
            starts.append(n - WINDOW_SENTS)
        windows += [(doc_index, s, min(s + WINDOW_SENTS, n)) for s in starts]
    return windows

label_counts = Counter(label for d in train_docs for label in d["labels"])
n_train_sents = sum(label_counts.values())
class_weights = torch.ones(NUM_LABELS)
for label_id in range(NUM_LABELS):
    if label_counts[label_id] > 0:
        w = (n_train_sents / (NUM_LABELS * label_counts[label_id])) ** CLASS_WEIGHT_POWER
        class_weights[label_id] = min(w, CLASS_WEIGHT_MAX)
CLASS_WEIGHTS = class_weights.to(DEVICE)

print(f"Stage 1 windows per epoch: {len(make_windows(train_docs))} (train)   "
      f"Stage 2: {len(make_windows(train_docs + dev_docs))} (train + dev)")

In [ ]:
# Cell 10 — Prediction (whole documents, Viterbi decoding) and evaluation.
# macro_f1 averages over all 13 labels (selects the epoch); macro_f1_present over labels that occur.
ALL_LABELS = list(range(NUM_LABELS))

@torch.no_grad()
def predict_docs(model, docs, decode="viterbi"):
    model.eval()
    all_preds = []
    for d in docs:
        vectors = []
        for start in range(0, len(d["ids"]), ENCODE_CHUNK):
            ids, mask = pad_batch(d["ids"][start:start + ENCODE_CHUNK])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                vectors.append(model.encode_sentences(ids.to(DEVICE), mask.to(DEVICE)).float())
        vectors = torch.cat(vectors)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            scores = model.emissions(vectors, rel_positions(len(d["ids"])).to(DEVICE), DOMAINS[d["domain"]])
        scores = scores.float()
        all_preds.append(model.crf.viterbi(scores) if decode == "viterbi" else scores.argmax(-1).tolist())
    return all_preds

def macro_scores(gold, pred):
    return {"macro_f1": f1_score(gold, pred, labels=ALL_LABELS, average="macro", zero_division=0),
            "macro_p": precision_score(gold, pred, labels=ALL_LABELS, average="macro", zero_division=0),
            "macro_r": recall_score(gold, pred, labels=ALL_LABELS, average="macro", zero_division=0),
            "macro_f1_present": f1_score(gold, pred, average="macro", zero_division=0)}

def evaluate(model, docs, decode="viterbi"):
    preds = predict_docs(model, docs, decode)
    result = {}
    for domain in ("CL", "IT", "ALL"):
        gold = [y for d in docs if domain in (d["domain"], "ALL") for y in d["labels"]]
        pred = [y for d, p in zip(docs, preds) if domain in (d["domain"], "ALL") for y in p]
        scores = macro_scores(gold, pred)
        if domain == "ALL":
            result.update(scores)
        else:
            result[f"{domain}_macro_f1_present"] = scores["macro_f1_present"]
    return result, preds

print("Prediction and evaluation helpers ready.")

In [ ]:
# Cell 11 — Teacher: find S7's checkpoints, rebuild the teacher, and turn its scores into soft labels.
# Soft labels = softmax(teacher per-sentence scores / T), computed once per whole document.
def find_teacher_checkpoint(stage):
    found = [p for p in sorted(glob.glob(f"/kaggle/input/**/checkpoints/{stage}/best.pt", recursive=True))
             if RUN_NAME not in p]
    assert found, f"Add the dataset with S7's checkpoints/{stage}/best.pt as an input."
    return found[0]

def load_teacher(stage):
    path = find_teacher_checkpoint(stage)
    trained = torch.load(path, map_location="cpu", weights_only=False)["trainable"]
    teacher = build_teacher()
    expected = {n for n, p in teacher.named_parameters() if p.requires_grad}
    assert set(trained) == expected, ("This checkpoint is not an S7 (LoRA + CRF) model: "
                                      f"{len(set(trained) ^ expected)} parameter names differ. Check the teacher dataset.")
    teacher.load_state_dict(trained, strict=False)
    teacher.eval()
    print(f"Teacher ({stage}) loaded from {path}")
    return teacher

@torch.no_grad()
def teacher_soft_labels(teacher, docs):
    soft = []
    for d in docs:
        vectors = []
        for start in range(0, len(d["ids"]), ENCODE_CHUNK):
            ids, mask = pad_batch(d["ids"][start:start + ENCODE_CHUNK])
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                vectors.append(teacher.encode_sentences(ids.to(DEVICE), mask.to(DEVICE)).float())
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            scores = teacher.emissions(torch.cat(vectors), rel_positions(len(d["ids"])).to(DEVICE), DOMAINS[d["domain"]])
        soft.append(F.softmax(scores.float() / KD_TEMPERATURE, dim=-1).cpu())
    return soft

def timed_predict(model, docs, decode):
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    preds = predict_docs(model, docs, decode)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return preds, time.time() - start

print("Teacher helpers ready.")

In [ ]:
# Cell 12 — train_stage(): student training with the KD loss, per-epoch checkpoints and automatic resume.
# Stage 1 (select_on_dev=True) scores dev with Viterbi (CRF) and with plain argmax every epoch and keeps the
# best (epoch, decoding) by dev Macro-F1, so the CRF is only used if it helps; Stage 2 trains a fixed number of epochs.
def trainable_state(model):
    full = model.state_dict()
    return {n: full[n].detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def train_stage(stage, docs, soft_targets, epochs_to_run, select_on_dev):
    stage_dir = CKPT_DIR / stage
    latest_path, best_path = stage_dir / "latest.pt", stage_dir / "best.pt"
    # Copy this stage's checkpoints from an attached earlier run of this notebook, if there is one.
    if not latest_path.exists():
        previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/{stage}/latest.pt", recursive=True))
        if previous:
            shutil.copy(previous[0], latest_path)
            if Path(previous[0]).with_name("best.pt").exists():
                shutil.copy(Path(previous[0]).with_name("best.pt"), best_path)
            print("Copied", stage, "checkpoints from", Path(previous[0]).parent)

    model = build_model()
    adapter = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
    crf = list(model.crf.parameters())
    head = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n and not n.startswith("crf.")]
    optimizer = torch.optim.AdamW([{"params": adapter, "lr": LR_ADAPTER}, {"params": head, "lr": LR_HEAD},
                                   {"params": crf, "lr": LR_CRF, "weight_decay": 0.0}], weight_decay=WEIGHT_DECAY)
    windows = make_windows(docs)
    total_steps = EPOCHS * len(windows)
    warmup_steps = int(WARMUP_FRACTION * total_steps)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: (step + 1) / max(warmup_steps, 1)
        if step < warmup_steps else max(0.0, (total_steps - step) / max(total_steps - warmup_steps, 1)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    state = {"epoch": 0, "best_f1": -1.0, "best_epoch": 0, "best_decode": "viterbi", "bad_epochs": 0, "history": [], "train_seconds": 0.0}
    if latest_path.exists():
        ckpt = torch.load(latest_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["trainable"], strict=False)
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        state = ckpt["state"]
        print(f"[{stage}] resumed after epoch {state['epoch']}")

    for epoch in range(state["epoch"], epochs_to_run):
        if select_on_dev and state["bad_epochs"] >= PATIENCE:
            print(f"[{stage}] early stopping: no dev improvement for {PATIENCE} epochs.")
            break
        model.train()
        order = windows[:]
        random.Random(SEED + epoch).shuffle(order)
        start_time, loss_sum = time.time(), 0.0
        for doc_index, s, e in order:
            d = docs[doc_index]
            ids, mask = pad_batch(d["ids"][s:e])
            labels = torch.tensor(d["labels"][s:e], device=DEVICE)
            positions = rel_positions(len(d["ids"]))[s:e].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                vectors = model.encode_sentences(ids.to(DEVICE), mask.to(DEVICE))
                scores = model.emissions(vectors, positions, DOMAINS[d["domain"]])
            scores = scores.float()
            teacher_probs = soft_targets[doc_index][s:e].to(DEVICE)
            kd = F.kl_div(F.log_softmax(scores / KD_TEMPERATURE, dim=-1), teacher_probs,
                          reduction="batchmean") * KD_TEMPERATURE ** 2
            loss = model.crf.neg_log_likelihood(scores, labels) / len(labels) \
                   + CE_WEIGHT * F.cross_entropy(scores, labels, weight=CLASS_WEIGHTS) + KD_WEIGHT * kd
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(adapter + head + crf, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item()

        seconds = time.time() - start_time
        state["train_seconds"] += seconds
        state["epoch"] = epoch + 1
        record = {"epoch": epoch + 1, "train_loss": loss_sum / len(order), "seconds": seconds}
        message = f"[{stage}] epoch {epoch + 1:2d} | loss {record['train_loss']:.4f}"
        if select_on_dev:
            dev_viterbi, _ = evaluate(model, dev_docs, "viterbi")
            dev_argmax, _ = evaluate(model, dev_docs, "argmax")
            decode = "viterbi" if dev_viterbi["macro_f1"] >= dev_argmax["macro_f1"] else "argmax"
            dev_scores = dev_viterbi if decode == "viterbi" else dev_argmax
            record.update(dev_scores, decode=decode, macro_f1_viterbi=dev_viterbi["macro_f1"],
                          macro_f1_argmax=dev_argmax["macro_f1"])
            message += (f" | dev Macro-F1 Viterbi {dev_viterbi['macro_f1']:.4f} / argmax {dev_argmax['macro_f1']:.4f} "
                        f"(best: present-labels {dev_scores['macro_f1_present']:.4f}; "
                        f"CL {dev_scores['CL_macro_f1_present']:.4f}, IT {dev_scores['IT_macro_f1_present']:.4f})")
            if dev_scores["macro_f1"] > state["best_f1"]:
                state.update(best_f1=dev_scores["macro_f1"], best_epoch=epoch + 1, best_decode=decode, bad_epochs=0)
                torch.save({"trainable": trainable_state(model), "epoch": epoch + 1, "decode": decode,
                            "dev": dev_scores}, best_path)
                message += f"  <- best ({decode})"
            else:
                state["bad_epochs"] += 1
        state["history"].append(record)
        torch.save({"trainable": trainable_state(model), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "state": state}, latest_path)
        print(message + f" | {seconds / 60:.1f} min")

    if not select_on_dev:
        torch.save({"trainable": trainable_state(model), "epoch": state["epoch"]}, best_path)
    return model, state

n_trainable = sum(p.numel() for p in build_model().parameters() if p.requires_grad)
torch.cuda.empty_cache()
print(f"Trainable parameters per student: {n_trainable:,}")

In [ ]:
# Cell 13 — Stage 1: S7 Stage-1 teacher → soft labels on train; teacher dev score; train the student on train.
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
teacher = load_teacher("stage1")
n_teacher_total = sum(p.numel() for p in teacher.parameters())
teacher_dev_viterbi, _ = evaluate(teacher, dev_docs, "viterbi")
_, teacher_dev_seconds = timed_predict(teacher, dev_docs, "viterbi")
soft_train = teacher_soft_labels(teacher, train_docs)
del teacher
torch.cuda.empty_cache()
print(f"Teacher (S7 Stage 1) dev Macro-F1 (13 labels, Viterbi) {teacher_dev_viterbi['macro_f1']:.4f}; "
      f"labels the dev set in {teacher_dev_seconds:.1f} s")

student1, stage1_state = train_stage("stage1", train_docs, soft_train, EPOCHS, select_on_dev=True)
BEST_EPOCH = stage1_state["best_epoch"]
BEST_DECODE = stage1_state["best_decode"]
print(f"Student Stage 1 best dev Macro-F1 (13 labels) {stage1_state['best_f1']:.4f} at epoch {BEST_EPOCH} "
      f"with {BEST_DECODE} decoding")

In [ ]:
# Cell 14 — Student Stage-1 best model: dev report, CRF ablation, teacher-vs-student size/speed, test predictions.
best1 = torch.load(CKPT_DIR / "stage1" / "best.pt", map_location="cpu", weights_only=False)
student1.load_state_dict(best1["trainable"], strict=False)
dev_scores_viterbi, _ = evaluate(student1, dev_docs, "viterbi")
dev_scores_argmax, _ = evaluate(student1, dev_docs, "argmax")
dev_scores, dev_preds = evaluate(student1, dev_docs, BEST_DECODE)
_, student_dev_seconds = timed_predict(student1, dev_docs, BEST_DECODE)
n_student_total = sum(p.numel() for p in student1.parameters())

gold_flat = [y for d in dev_docs for y in d["labels"]]
pred_flat = [y for p in dev_preds for y in p]
report = classification_report(gold_flat, pred_flat, labels=ALL_LABELS,
                               target_names=[ID2LABEL[i] for i in ALL_LABELS], zero_division=0)
print(f"Student Stage 1 best epoch {best1['epoch']} ({BEST_DECODE}) — dev Macro-F1 {dev_scores['macro_f1']:.4f} "
      f"(13 labels) | P {dev_scores['macro_p']:.4f} | R {dev_scores['macro_r']:.4f} | "
      f"present-labels {dev_scores['macro_f1_present']:.4f}")
print(f"CRF ablation — student dev Macro-F1 with Viterbi {dev_scores_viterbi['macro_f1']:.4f} "
      f"vs plain argmax {dev_scores_argmax['macro_f1']:.4f}")
print(f"Teacher vs student — dev Macro-F1 {teacher_dev_viterbi['macro_f1']:.4f} vs {dev_scores['macro_f1']:.4f} "
      f"({100 * dev_scores['macro_f1'] / teacher_dev_viterbi['macro_f1']:.1f}% kept) | parameters "
      f"{n_teacher_total:,} vs {n_student_total:,} ({100 * n_student_total / n_teacher_total:.1f}%) | "
      f"dev labelling time {teacher_dev_seconds:.1f} s vs {student_dev_seconds:.1f} s "
      f"({teacher_dev_seconds / student_dev_seconds:.2f}× faster)")
print(report)

stage1_test_preds = predict_docs(student1, test_docs, BEST_DECODE)
cm = confusion_matrix(gold_flat, pred_flat, labels=ALL_LABELS)
json.dump({"dev_scores": dev_scores, "decode": BEST_DECODE, "dev_scores_viterbi": dev_scores_viterbi,
           "dev_scores_argmax": dev_scores_argmax, "teacher_dev_viterbi": teacher_dev_viterbi,
           "best_epoch": best1["epoch"], "history": stage1_state["history"], "confusion_matrix": cm.tolist(),
           "labels": [ID2LABEL[i] for i in ALL_LABELS]},
          open(OUT_DIR / "stage1_dev_metrics.json", "w"), indent=2)
open(OUT_DIR / "stage1_dev_classification_report.txt", "w").write(report)
del student1, soft_train
torch.cuda.empty_cache()
print("Saved Stage-1 metrics and kept the student's test predictions.")

In [ ]:
# Cell 15 — Stage 2: S7 Stage-2 teacher → soft labels on train + dev; retrain the student on train + dev.
# Same schedule as Stage 1, stopped at the student's Stage-1 best epoch.
teacher = load_teacher("stage2")
soft_train_dev = teacher_soft_labels(teacher, train_docs + dev_docs)
del teacher
torch.cuda.empty_cache()
student2, stage2_state = train_stage("stage2", train_docs + dev_docs, soft_train_dev, BEST_EPOCH, select_on_dev=False)
stage2_test_preds = predict_docs(student2, test_docs, BEST_DECODE)
print(f"Student Stage 2 finished after {stage2_state['epoch']} epochs (= Stage-1 best epoch {BEST_EPOCH}).")

In [ ]:
# Cell 16 — Write, validate and zip both submissions, then save the run summary.
# submission.zip = student Stage 2 (train + dev); submission_stage1.zip = student Stage 1 (train only).
expected_ids = set(json.load(open(P3_DIR / "Test/CL_test.json"))) | set(json.load(open(P3_DIR / "Test/IT_test.json")))

def write_submission(preds, name):
    prediction = {d["id"]: [ID2LABEL[y] for y in p] for d, p in zip(test_docs, preds)}
    assert set(prediction) == expected_ids, "Test document ids do not match"
    for d in test_docs:
        assert len(prediction[d["id"]]) == len(d["sents"]), f"Sentence count mismatch for {d['id']}"
    assert all(label in LABEL2ID for labels in prediction.values() for label in labels), "Invalid label found"
    folder = WORK_DIR / name
    folder.mkdir(exist_ok=True)
    json.dump(prediction, open(folder / "prediction.json", "w"), indent=2)
    zip_path = WORK_DIR / f"{name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(folder / "prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as zf:
        assert zf.namelist() == ["prediction.json"], "prediction.json must be at the zip root"
    counts = Counter(l for labels in prediction.values() for l in labels)
    print(f"{zip_path.name}: {len(prediction)} documents, {sum(counts.values())} sentences — "
          f"top labels {dict(counts.most_common(5))}")

write_submission(stage2_test_preds, "submission")
write_submission(stage1_test_preds, "submission_stage1")

summary = {
    "run": RUN_NAME, "technique": "Knowledge distillation: S7 teacher (12 layers) -> 6-layer student, LoRA + BiLSTM-CRF",
    "student_layers": list(STUDENT_LAYERS), "kd_temperature": KD_TEMPERATURE, "kd_weight": KD_WEIGHT,
    "teacher_total_params": n_teacher_total, "student_total_params": n_student_total,
    "student_trainable_params": n_trainable,
    "teacher_dev_seconds": round(teacher_dev_seconds, 1), "student_dev_seconds": round(student_dev_seconds, 1),
    "stage1_best_epoch": BEST_EPOCH, "decode": BEST_DECODE, "stage1_epochs_run": stage1_state["epoch"],
    "stage1_minutes": round(stage1_state["train_seconds"] / 60, 1),
    "stage2_minutes": round(stage2_state["train_seconds"] / 60, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "teacher_dev": teacher_dev_viterbi, "student_dev": dev_scores,
    "student_dev_viterbi": dev_scores_viterbi, "student_dev_argmax": dev_scores_argmax,
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))